# Mode A: Baseline Pure NSGA-II

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_data()`, `run_nsga2()`, `plot_convergence()` |
| `src/schedule_engine/config/models.py` | Global time config | `quantum_minutes`, `opening_time` |
| **This notebook** | Mode-specific config | `POP_SIZE`, `NGEN`, experiment execution |

## Mode A: Pure NSGA-II
- No repair heuristics
- No local search
- No RL guidance
- Baseline for comparison

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
import random
import numpy as np
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks.core import load_data, create_random_individual
from schedule_engine.notebooks.core import course_aware_crossover, smart_mutation
from schedule_engine.notebooks.core import create_evaluator, get_constraint_breakdown
from schedule_engine.notebooks.core import run_nsga2, EvolutionConfig, get_best_individual
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Mode A Configuration (Inline - Mode-Specific)

In [ ]:
# ============================================================================
# MODE A CONFIGURATION - Modify these as needed
# ============================================================================
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Evolution config (inline - mode-specific)
config = EvolutionConfig(
    pop_size=50,
    ngen=100,
    cxpb=0.9,
    mutpb=0.2,
    fitness_weights=(-1.0, -0.01),  # (hard, soft) - both minimized
    verbose=True,
    log_interval=20,
)

# Paths - Organized by mode with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_a_baseline/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Mode A Config: pop={config.pop_size}, ngen={config.ngen}, cxpb={config.cxpb}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data (using `schedule_engine/notebooks/data_loader`)

In [ ]:
# Load all data with single function call
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

## 4. Test Population & Evaluation

In [ ]:
# Test individual creation
test_ind = create_random_individual(data)
print(f" Individual has {len(test_ind)} genes")

# Test evaluation
evaluate = create_evaluator(data)
test_fitness = evaluate(test_ind)
print(f" Test fitness: hard={test_fitness[0]}, soft={test_fitness[1]}")

## 5. Run NSGA-II Evolution

In [ ]:
# Run evolution with DRY components
final_pop, stats = run_nsga2(
    data=data,
    config=config,
    create_individual_fn=create_random_individual,
    evaluate_fn=evaluate,
    crossover_fn=course_aware_crossover,
    mutate_fn=lambda ind: smart_mutation(ind, data),  # Closure over data
)

## 6. Results & Visualization

In [ ]:
# Get best solution
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

# Print summary
print_summary(final_pop, stats, breakdown)

# Plot results
plot_convergence(stats, OUTPUT_DIR / "mode_a_convergence.png", title_prefix="Mode A: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_a_breakdown.png", title="Mode A: Constraint Violations")

## 7. Full Production Export (Optional)

Run this cell to generate the same outputs as CLI production runs:
- `schedule.json` - Full decoded schedule
- `calendar.pdf` - Visual calendar
- `plots/constraints/` - Constraint trend plots
- `plots/nsga/` - NSGA metrics plots
- `csv/` - Evolution data CSVs

In [ ]:
# Reload export module with fixed inline decode
import importlib
import schedule_engine.notebooks.export
importlib.reload(schedule_engine.notebooks.export)

from schedule_engine.notebooks.export import export_full_results

# Generate all production outputs (same as CLI)
export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_a_baseline",
)

# Show output location
print(f"\n All files saved to: {export_paths['output_dir']}")